# Download SGK PDF - Lớp 10, 11, 12 từ thi247.com

Notebook tự động scrape và download tất cả PDF sách giáo khoa từ thi247.com

In [1]:
# Cell 1: Dependencies already installed
print('Dependencies ready: requests, beautifulsoup4, lxml')

Dependencies ready: requests, beautifulsoup4, lxml


In [3]:
# Cell 2: Imports and configuration
import requests
from bs4 import BeautifulSoup
import os
import time
import re
from pathlib import Path
from urllib.parse import urlparse, unquote

# Configuration
BASE_URL = 'https://thi247.com'
CATEGORIES = {
    'lop-10': {'url': f'{BASE_URL}/chuyen-muc/sach-giao-khoa-lop-10/', 'pages': 6},
    'lop-11': {'url': f'{BASE_URL}/chuyen-muc/sach-giao-khoa-lop-11/', 'pages': 9},
    'lop-12': {'url': f'{BASE_URL}/chuyen-muc/sach-giao-khoa-lop-12/', 'pages': 7},
}
# Use project root data/ folder
PROJECT_ROOT = Path('/home/sakana/Code/ManifoldsTeam/AIServices')
DATA_DIR = PROJECT_ROOT / 'data'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36'
}

# Create output directories
for cat in CATEGORIES:
    (DATA_DIR / cat).mkdir(parents=True, exist_ok=True)
    
print(f'Data directory: {DATA_DIR}')
print(f'Subdirectories created: {list(CATEGORIES.keys())}')

Data directory: /home/sakana/Code/ManifoldsTeam/AIServices/data
Subdirectories created: ['lop-10', 'lop-11', 'lop-12']


In [4]:
# Cell 3: Scrape all article URLs from all category pages
def get_article_links(category_url, num_pages):
    """Scrape all article links from category listing pages."""
    links = []
    for page in range(1, num_pages + 1):
        if page == 1:
            url = category_url
        else:
            url = f'{category_url}page/{page}/'
        
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            resp.raise_for_status()
            soup = BeautifulSoup(resp.text, 'lxml')
            
            # Find article links - they're in h2 tags within the main content
            for h2 in soup.find_all('h2', class_='entry-title'):
                a = h2.find('a')
                if a and a.get('href'):
                    links.append({
                        'title': a.get_text(strip=True),
                        'url': a['href']
                    })
            
            print(f'  Page {page}/{num_pages}: found {len(links)} total links')
            time.sleep(0.5)  # Be polite
        except Exception as e:
            print(f'  Error on page {page}: {e}')
    
    return links

# Collect all article links
all_articles = {}
for cat_name, cat_info in CATEGORIES.items():
    print(f'\nScraping {cat_name}...')
    articles = get_article_links(cat_info['url'], cat_info['pages'])
    all_articles[cat_name] = articles
    print(f'  Total articles for {cat_name}: {len(articles)}')

total = sum(len(v) for v in all_articles.values())
print(f'\n=== TOTAL ARTICLES FOUND: {total} ===')


Scraping lop-10...
  Page 1/6: found 12 total links
  Page 2/6: found 24 total links
  Page 3/6: found 36 total links
  Page 4/6: found 48 total links
  Page 5/6: found 60 total links
  Page 6/6: found 66 total links
  Total articles for lop-10: 66

Scraping lop-11...
  Page 1/9: found 12 total links
  Page 2/9: found 24 total links
  Page 3/9: found 36 total links
  Page 4/9: found 48 total links
  Page 5/9: found 60 total links
  Page 6/9: found 72 total links
  Page 7/9: found 84 total links
  Page 8/9: found 96 total links
  Page 9/9: found 101 total links
  Total articles for lop-11: 101

Scraping lop-12...
  Page 1/7: found 12 total links
  Page 2/7: found 24 total links
  Page 3/7: found 36 total links
  Page 4/7: found 48 total links
  Page 5/7: found 60 total links
  Page 6/7: found 72 total links
  Page 7/7: found 75 total links
  Total articles for lop-12: 75

=== TOTAL ARTICLES FOUND: 242 ===


In [5]:
# Cell 4: Extract PDF URLs from article pages
# Pattern: PDF is embedded via wonderplugin-pdf-embed viewer
# Direct PDF URL: https://thi247.com/thi247-pdf/<slug>.pdf

def extract_pdf_url(article_url):
    """Visit article page and extract the direct PDF URL."""
    try:
        resp = requests.get(article_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        
        # Method 1: Find PDF URL in the wonderplugin iframe/embed
        # Pattern: file=https://thi247.com/thi247-pdf/xxx.pdf
        match = re.search(r'file=(https?://[^"\s&#]+\.pdf)', resp.text)
        if match:
            return unquote(match.group(1))
        
        # Method 2: Direct link to PDF
        soup = BeautifulSoup(resp.text, 'lxml')
        for a in soup.find_all('a', href=True):
            if a['href'].endswith('.pdf'):
                return a['href']
        
        # Method 3: Guess from slug
        slug = article_url.rstrip('/').split('/')[-1]
        guessed = f'{BASE_URL}/thi247-pdf/{slug}.pdf'
        # Verify with HEAD request
        head = requests.head(guessed, headers=HEADERS, timeout=10, allow_redirects=True)
        if head.status_code == 200:
            return guessed
        
        return None
    except Exception as e:
        return None

# Test with one article first
test_url = all_articles['lop-12'][0]['url']
test_pdf = extract_pdf_url(test_url)
print(f'Test article: {test_url}')
print(f'PDF URL: {test_pdf}')

Test article: https://thi247.com/chuyen-de-hoc-tap-ngu-van-12-canh-dieu/
PDF URL: https://thi247.com/thi247-pdf/chuyen-de-hoc-tap-ngu-van-12-canh-dieu.pdf


In [6]:
# Cell 5: Build PDF URL list for all articles
# Since we found the pattern, we can directly construct PDF URLs from slugs
# But we'll verify each by visiting the page to be safe
# cần phân chia theo từng môn đối với mỗi khối trong folder 
pdf_downloads = {}  # {category: [{title, article_url, pdf_url, filename}]}

for cat_name, articles in all_articles.items():
    print(f'\nProcessing {cat_name} ({len(articles)} articles)...')
    pdf_downloads[cat_name] = []
    
    for i, article in enumerate(articles):
        slug = article['url'].rstrip('/').split('/')[-1]
        pdf_url = f'{BASE_URL}/thi247-pdf/{slug}.pdf'
        filename = f'{slug}.pdf'
        
        pdf_downloads[cat_name].append({
            'title': article['title'],
            'article_url': article['url'],
            'pdf_url': pdf_url,
            'filename': filename
        })
        
        if (i + 1) % 10 == 0:
            print(f'  Processed {i+1}/{len(articles)}')
    
    print(f'  Done: {len(pdf_downloads[cat_name])} PDFs queued')

total_pdfs = sum(len(v) for v in pdf_downloads.values())
print(f'\n=== TOTAL PDFs TO DOWNLOAD: {total_pdfs} ===')


Processing lop-10 (66 articles)...
  Processed 10/66
  Processed 20/66
  Processed 30/66
  Processed 40/66
  Processed 50/66
  Processed 60/66
  Done: 66 PDFs queued

Processing lop-11 (101 articles)...
  Processed 10/101
  Processed 20/101
  Processed 30/101
  Processed 40/101
  Processed 50/101
  Processed 60/101
  Processed 70/101
  Processed 80/101
  Processed 90/101
  Processed 100/101
  Done: 101 PDFs queued

Processing lop-12 (75 articles)...
  Processed 10/75
  Processed 20/75
  Processed 30/75
  Processed 40/75
  Processed 50/75
  Processed 60/75
  Processed 70/75
  Done: 75 PDFs queued

=== TOTAL PDFs TO DOWNLOAD: 242 ===


In [7]:
# Cell 6: Download all PDFs
def download_pdf(pdf_url, save_path, max_retries=3):
    """Download a PDF file with retry logic."""
    if save_path.exists() and save_path.stat().st_size > 1000:
        return 'skipped'  # Already downloaded
    
    for attempt in range(max_retries):
        try:
            resp = requests.get(pdf_url, headers=HEADERS, timeout=120, stream=True)
            if resp.status_code == 200:
                content_type = resp.headers.get('content-type', '')
                # Check if response is actually PDF
                with open(save_path, 'wb') as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        f.write(chunk)
                size_kb = save_path.stat().st_size / 1024
                if size_kb < 1:  # Likely an error page
                    save_path.unlink()
                    return 'too_small'
                return 'ok'
            elif resp.status_code == 404:
                return '404'
            else:
                if attempt < max_retries - 1:
                    time.sleep(2 ** attempt)
                continue
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            continue
    return 'failed'

# Download all PDFs
results = {'ok': 0, 'skipped': 0, '404': 0, 'failed': 0, 'too_small': 0}
failed_list = []

for cat_name, pdfs in pdf_downloads.items():
    print(f'\n{"="*60}')
    print(f'Downloading {cat_name} ({len(pdfs)} files)...')
    print(f'{"="*60}')
    
    for i, pdf_info in enumerate(pdfs):
        save_path = DATA_DIR / cat_name / pdf_info['filename']
        status = download_pdf(pdf_info['pdf_url'], save_path)
        results[status] += 1
        
        if status == 'ok':
            size_mb = save_path.stat().st_size / (1024 * 1024)
            print(f'  [{i+1}/{len(pdfs)}] ✅ {pdf_info["filename"]} ({size_mb:.1f} MB)')
        elif status == 'skipped':
            print(f'  [{i+1}/{len(pdfs)}] ⏭ {pdf_info["filename"]} (already exists)')
        elif status == '404':
            failed_list.append(pdf_info)
            print(f'  [{i+1}/{len(pdfs)}] ❌ {pdf_info["filename"]} (404 NOT FOUND)')
        elif status == 'too_small':
            failed_list.append(pdf_info)
            print(f'  [{i+1}/{len(pdfs)}] ⚠️ {pdf_info["filename"]} (too small, likely error page)')
        else:
            failed_list.append(pdf_info)
            print(f'  [{i+1}/{len(pdfs)}] ❌ {pdf_info["filename"]} (FAILED)')
        
        time.sleep(0.3)  # Rate limiting

print(f'\n{"="*60}')
print(f'DOWNLOAD COMPLETE')
print(f'{"="*60}')
print(f'✅ OK: {results["ok"]}')
print(f'⏭ Skipped: {results["skipped"]}')
print(f'❌ 404: {results["404"]}')
print(f'⚠️ Too small: {results["too_small"]}')
print(f'❌ Failed: {results["failed"]}')


  [1/66] ✅ chuyen-de-hoc-tap-dia-li-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (6.6 MB)
  [2/66] ✅ chuyen-de-hoc-tap-gdktpl-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (14.8 MB)
  [3/66] ✅ chuyen-de-hoc-tap-toan-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (9.9 MB)
  [4/66] ✅ sach-giao-khoa-toan-10-tap-2-chan-troi-sang-tao.pdf (7.5 MB)
  [5/66] ✅ sach-giao-khoa-toan-10-tap-1-chan-troi-sang-tao.pdf (10.6 MB)
  [6/66] ✅ sach-giao-khoa-gdktpl-10-chan-troi-sang-tao.pdf (16.0 MB)
  [7/66] ✅ sach-giao-khoa-dia-li-10-chan-troi-sang-tao.pdf (14.6 MB)
  [8/66] ✅ chuyen-de-hoc-tap-toan-10-chan-troi-sang-tao.pdf (5.2 MB)
  [9/66] ✅ chuyen-de-hoc-tap-lich-su-10-chan-troi-sang-tao.pdf (6.8 MB)
  [10/66] ✅ chuyen-de-hoc-tap-dia-li-10-chan-troi-sang-tao.pdf (3.1 MB)
  [11/66] ✅ chuyen-de-hoc-tap-gdktpl-10-chan-troi-sang-tao.pdf (6.9 MB)
  [12/66] ✅ chuyen-de-hoc-tap-lich-su-10-canh-dieu.pdf (22.7 MB)
  [13/66] ✅ chuyen-de-hoc-tap-lich-su-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (15.1 MB)
  [14/66] ✅ sach-giao-khoa-lich-

In [8]:
# Cell 7: Handle failed downloads - try extracting PDF URL from article page
if failed_list:
    print(f'Retrying {len(failed_list)} failed downloads by visiting article pages...\n')
    retry_results = {'ok': 0, 'failed': 0}
    still_failed = []
    
    for i, pdf_info in enumerate(failed_list):
        pdf_url = extract_pdf_url(pdf_info['article_url'])
        if pdf_url:
            # Determine category from article URL
            cat = None
            for cat_name in CATEGORIES:
                grade = cat_name.split('-')[1]  # '10', '11', '12'
                if f'-{grade}-' in pdf_info['filename'] or pdf_info['filename'].endswith(f'-{grade}.pdf'):
                    cat = cat_name
                    break
            if not cat:  # Fallback: check which category this article belongs to
                for cn, articles in all_articles.items():
                    if any(a['url'] == pdf_info['article_url'] for a in articles):
                        cat = cn
                        break
            if not cat:
                cat = 'lop-12'  # default
            
            # Extract filename from actual PDF URL
            actual_filename = pdf_url.split('/')[-1]
            save_path = DATA_DIR / cat / actual_filename
            
            status = download_pdf(pdf_url, save_path)
            if status == 'ok' or status == 'skipped':
                retry_results['ok'] += 1
                size_mb = save_path.stat().st_size / (1024*1024) if save_path.exists() else 0
                print(f'  [{i+1}/{len(failed_list)}] ✅ {actual_filename} ({size_mb:.1f} MB)')
            else:
                retry_results['failed'] += 1
                still_failed.append(pdf_info)
                print(f'  [{i+1}/{len(failed_list)}] ❌ {pdf_info["title"]} -> {pdf_url} ({status})')
        else:
            retry_results['failed'] += 1
            still_failed.append(pdf_info)
            print(f'  [{i+1}/{len(failed_list)}] ❌ No PDF found: {pdf_info["title"]}')
        
        time.sleep(1)
    
    print(f'\nRetry results: ✅ {retry_results["ok"]} / ❌ {retry_results["failed"]}')
    if still_failed:
        print(f'\nStill failed ({len(still_failed)}):')
        for f in still_failed:
            print(f'  - {f["title"]}: {f["article_url"]}')
else:
    print('All downloads successful! No retries needed.')

All downloads successful! No retries needed.


In [9]:
# Cell 8: Summary - count files and total size
print('=== FINAL SUMMARY ===')
total_files = 0
total_size = 0

for cat_name in CATEGORIES:
    cat_dir = DATA_DIR / cat_name
    files = list(cat_dir.glob('*.pdf'))
    cat_size = sum(f.stat().st_size for f in files)
    total_files += len(files)
    total_size += cat_size
    print(f'\n{cat_name}:')
    print(f'  Files: {len(files)}')
    print(f'  Size: {cat_size / (1024*1024):.1f} MB')
    for f in sorted(files):
        print(f'    - {f.name} ({f.stat().st_size / (1024*1024):.1f} MB)')

print(f'\n{"="*40}')
print(f'TOTAL: {total_files} PDFs, {total_size / (1024*1024):.1f} MB')

=== FINAL SUMMARY ===

lop-10:
  Files: 66
  Size: 1295.8 MB
    - chuyen-de-hoc-tap-dia-li-10-chan-troi-sang-tao.pdf (3.1 MB)
    - chuyen-de-hoc-tap-dia-li-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (6.6 MB)
    - chuyen-de-hoc-tap-gdktpl-10-chan-troi-sang-tao.pdf (6.9 MB)
    - chuyen-de-hoc-tap-gdktpl-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (14.8 MB)
    - chuyen-de-hoc-tap-hoa-hoc-10-canh-dieu.pdf (23.7 MB)
    - chuyen-de-hoc-tap-hoa-hoc-10-chan-troi-sang-tao.pdf (10.3 MB)
    - chuyen-de-hoc-tap-hoa-hoc-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (15.8 MB)
    - chuyen-de-hoc-tap-lich-su-10-canh-dieu.pdf (22.7 MB)
    - chuyen-de-hoc-tap-lich-su-10-chan-troi-sang-tao.pdf (6.8 MB)
    - chuyen-de-hoc-tap-lich-su-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (15.1 MB)
    - chuyen-de-hoc-tap-ngu-van-10-canh-dieu.pdf (17.2 MB)
    - chuyen-de-hoc-tap-ngu-van-10-chan-troi-sang-tao.pdf (26.2 MB)
    - chuyen-de-hoc-tap-ngu-van-10-ket-noi-tri-thuc-voi-cuoc-song.pdf (23.5 MB)
    - chuyen-de-hoc-tap-sinh-h

In [1]:
# Cell 9: Reorganize PDFs into subject folders per grade
import shutil
from pathlib import Path

DATA_DIR = Path('/home/sakana/Code/ManifoldsTeam/AIServices/data')

# Subject mapping: keyword in filename -> folder name
SUBJECT_MAP = [
    # Order matters - more specific patterns first
    ('dai-so-va-giai-tich', 'toan'),
    ('dai-so', 'toan'),
    ('giai-tich', 'toan'),
    ('hinh-hoc', 'toan'),
    ('toan', 'toan'),
    ('ngu-van', 'ngu-van'),
    ('vat-li', 'vat-li'),
    ('hoa-hoc', 'hoa-hoc'),
    ('sinh-hoc', 'sinh-hoc'),
    ('lich-su', 'lich-su'),
    ('dia-li', 'dia-li'),
    ('tieng-anh', 'tieng-anh'),
    ('tin-hoc', 'tin-hoc'),
    ('cong-nghe', 'cong-nghe'),
    ('giao-duc-quoc-phong', 'gdqp-an'),
    ('giao-duc-cong-dan', 'gdcd'),
    ('gdktpl', 'gdcd'),
    ('hoat-dong-trai-nghiem', 'hoat-dong-trai-nghiem'),
]

def classify_subject(filename):
    """Classify a PDF filename into a subject folder."""
    name = filename.lower()
    for keyword, subject in SUBJECT_MAP:
        if keyword in name:
            return subject
    return 'khac'  # uncategorized

# Process each grade
for grade_dir in ['lop-10', 'lop-11', 'lop-12']:
    src_dir = DATA_DIR / grade_dir
    pdfs = list(src_dir.glob('*.pdf'))
    
    print(f'\n{"="*60}')
    print(f'{grade_dir.upper()} ({len(pdfs)} files)')
    print(f'{"="*60}')
    
    subject_counts = {}
    
    for pdf in pdfs:
        subject = classify_subject(pdf.name)
        dest_dir = src_dir / subject
        dest_dir.mkdir(exist_ok=True)
        
        dest_path = dest_dir / pdf.name
        shutil.move(str(pdf), str(dest_path))
        
        subject_counts[subject] = subject_counts.get(subject, 0) + 1
    
    # Print summary for this grade
    for subject in sorted(subject_counts.keys()):
        print(f'  {subject:25s} : {subject_counts[subject]} files')

print(f'\n{"="*60}')
print('DONE - All files reorganized!')
print(f'{"="*60}')


LOP-10 (66 files)
  cong-nghe                 : 1 files
  dia-li                    : 5 files
  gdcd                      : 4 files
  gdqp-an                   : 1 files
  hoa-hoc                   : 8 files
  lich-su                   : 7 files
  ngu-van                   : 13 files
  sinh-hoc                  : 8 files
  tieng-anh                 : 2 files
  tin-hoc                   : 1 files
  toan                      : 8 files
  vat-li                    : 8 files

LOP-11 (101 files)
  cong-nghe                 : 9 files
  dia-li                    : 8 files
  gdcd                      : 7 files
  gdqp-an                   : 1 files
  hoa-hoc                   : 8 files
  hoat-dong-trai-nghiem     : 3 files
  lich-su                   : 8 files
  ngu-van                   : 13 files
  sinh-hoc                  : 8 files
  tieng-anh                 : 6 files
  tin-hoc                   : 9 files
  toan                      : 13 files
  vat-li                    : 8 files

LOP-12 